# 5.3 — Lasso Regression
### Automatic Feature Selection with L1 Regularisation

---

## The Problem Ridge Couldn't Solve

Ridge Regression was great at taming wild coefficients when features are correlated. But it has one limitation:

**Ridge never eliminates features. It only shrinks them.**

Imagine Meena at her Mumbai real estate startup. She scraped 30 features for every flat — but honestly, most of them are noise:
- `flat_id` — just an ID number
- `agent_code` — which agent listed it
- `listing_day` — day of week it was listed
- `photo_count` — number of photos uploaded
- ...and 21 more just like these

Only 5 features actually matter: `area_sqft`, `distance_from_station_km`, `floor_number`, `age_years`, `locality_score`.

Ridge would keep all 30 features — just make the 25 useless ones very small. The model is still cluttered. Meena still can't tell her team which features to stop collecting.

**Lasso fixes this. It pushes useless coefficients to exactly zero — eliminating them entirely.**

---

## The One Change That Makes Everything Different

Compare the two penalties:

| | Loss Function |
|---|---|
| **Ridge** | $MSE + \lambda \sum w_i^2$ |
| **Lasso** | $MSE + \lambda \sum |w_i|$ |

Squared → Absolute value. That's the only change.

But this one change causes completely different behaviour.

---

## Why Absolute Value Reaches Zero (and Squared Never Does)

Think about the **slope** of each penalty:

**Ridge penalty = w²**
- Slope = 2w
- When w = 100 → push = 200 (huge)
- When w = 1 → push = 2 (small)
- When w = 0.001 → push = 0.002 (nearly gone)
- When w = 0 → push = **0** ← push disappears. Never reaches zero.

**Lasso penalty = |w|** (a V-shape)
- Slope = **1** always (constant — it's a straight line on each side)
- When w = 100 → push = 1
- When w = 1 → push = 1
- When w = 0.001 → push = 1
- The push **never weakens**. It keeps pushing with constant force all the way to zero — and kicks the coefficient across.

**Ridge runs out of steam. Lasso never does.**

This is why Lasso can set coefficients to **exactly zero** — it's a mathematical consequence of the constant slope of |w|.

---

## Automatic Feature Selection

When Lasso sets a coefficient to exactly zero:
- That feature is **completely removed** from the model
- It contributes nothing to any prediction
- The model becomes simpler and more interpretable

This is called **automatic feature selection** — Lasso decides which features matter and which don't, built into the loss function itself. No separate feature selection step needed.

---

## Lasso's Weakness — Correlated Features

When two features are correlated (e.g. `area_sqft` and `num_bedrooms`), both carry similar information.

- **Ridge:** keeps both, shrinks both — stable
- **Lasso:** picks one **arbitrarily**, zeros the other out

This is a problem because the choice is arbitrary — change your training data slightly and Lasso might keep the other one instead. Unstable and potentially misleading.

| Situation | Use |
|-----------|-----|
| Many features, most are noise | ✅ Lasso — eliminates useless ones |
| Many correlated features | ✅ Ridge — stabilises both |
| Both problems at once | ✅ ElasticNet (combines L1 + L2) |

---

## Finding the Best Alpha — Why Cross Validation?

Just like Ridge, Lasso has an `alpha` (= λ) that controls penalty strength.

We use **cross-validation** to find the best alpha — never the test set. Here's why:

| Data split | Used for |
|------------|----------|
| Training folds | Learning w and b (actual model weights) |
| Validation fold | Picking the best alpha only |
| Test set | Final honest evaluation — touched **once**, at the very end |

If you used the test set to pick alpha, you'd be indirectly making decisions based on it. Your final test RMSE would be optimistically biased — better than what you'd get on truly new data.

**5-fold CV:** Split training data into 5 folds. For each alpha candidate, train on 4 folds, validate on 1. Rotate. Average the 5 scores. Pick alpha with lowest average RMSE. Test set never touched.

`LassoCV` in sklearn does all of this automatically.

---

## The Lasso Coefficient Path

As alpha increases:
- Small alpha → most coefficients non-zero
- Medium alpha → noise features start hitting **exactly zero** and disappearing one by one
- Large alpha → only the strongest real features survive
- Very large alpha → everything goes to zero → useless model

Unlike Ridge (where all coefficients shrink smoothly but never touch zero), Lasso coefficients **drop off sharply** — each dropout = one feature eliminated. This creates a staircase-like path.

---

## Real World Problem — Mumbai Flat Price Prediction

**Meena** works at a real estate startup in Mumbai. She has data on 1000 flats with **30 features** — but only 5 of them actually drive the price. The other 25 are noise that crept in during data collection.

**She wants two things:**
1. Accurate price predictions
2. Know which features actually matter — so her team stops wasting time collecting useless data

**Why Lasso?** It will automatically zero out the 25 noise features and keep only the 5 real ones — giving Meena both things in one model.

**We will:**
1. Create 1000 flats with 5 real + 25 noise features
2. Compare Linear Regression vs Ridge vs Lasso — which features does each keep?
3. Verify Lasso correctly identifies the 5 real features
4. Find best alpha using LassoCV
5. Plot the Lasso coefficient path — watch features drop off one by one

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# WHY these imports:
# Lasso    — sklearn's Lasso implementation (uses coordinate descent internally)
# LassoCV  — Lasso with built-in cross-validation to find best alpha automatically
# Ridge    — for comparison against Lasso
# Pipeline — chains scaler + lasso so scaling never leaks into test data
# StandardScaler — mandatory before Lasso (same reason as Ridge — fair penalty)

np.random.seed(42)

In [ ]:
# ── Step 1: Create Dataset — 5 Real Features + 25 Noise Features ──────────────
n = 1000  # 1000 Mumbai flats

# ── 5 REAL features (these actually drive the price) ──
area_sqft             = np.random.randint(300, 2000, n)
distance_from_station = np.round(np.random.uniform(0.2, 10, n), 1)  # km
floor_number          = np.random.randint(1, 25, n)
age_years             = np.random.randint(0, 40, n)
locality_score        = np.random.randint(1, 11, n)  # 1-10

# ── 25 NOISE features (random numbers — zero real relationship with price) ──
# WHY create noise features?
# To simulate real-world messy data collection.
# We know the truth — so after Lasso runs, we can verify it correctly zeroed these out.
noise_features = np.random.randn(n, 25)
# np.random.randn gives standard normal random numbers — pure noise
noise_col_names = [f'noise_{i+1:02d}' for i in range(25)]

# ── True price formula (ONLY real features matter) ──
price_noise = np.random.normal(0, 3, n)
price_lakhs = (
    0.05  * area_sqft +          # bigger flat = higher price
   -2.0   * distance_from_station +  # further from station = cheaper
    0.4   * floor_number +       # higher floor = slightly more expensive
   -0.7   * age_years +          # older building = cheaper
    3.0   * locality_score +     # better locality = much more expensive
    15    +                      # base price
    price_noise
)

# ── Build DataFrame ──
real_df = pd.DataFrame({
    'area_sqft':             area_sqft,
    'distance_from_station': distance_from_station,
    'floor_number':          floor_number,
    'age_years':             age_years,
    'locality_score':        locality_score,
})

noise_df   = pd.DataFrame(noise_features, columns=noise_col_names)
df         = pd.concat([real_df, noise_df], axis=1)
# WHY concat? — combine real and noise features into one DataFrame
# axis=1 means combine column-wise (side by side), not row-wise

df['price_lakhs'] = np.round(price_lakhs, 2)

print(f"Dataset shape: {df.shape}")
print(f"Total features: {df.shape[1]-1} (5 real + 25 noise)")
print(f"\nReal features: {list(real_df.columns)}")
print(f"Noise features: {noise_col_names[:5]}... (25 total)")
print(f"\nAverage price: ₹{df['price_lakhs'].mean():.2f} lakhs")

In [ ]:
# ── Step 2: Split Data ────────────────────────────────────────────────────────
X = df.drop('price_lakhs', axis=1)  # 30 features
y = df['price_lakhs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape[0]} flats | Test: {X_test.shape[0]} flats")
print(f"Features given to each model: {X_train.shape[1]}")

In [ ]:
# ── Step 3: Train All Three Models ───────────────────────────────────────────

# ── Linear Regression ──
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# ── Ridge (alpha=1.0) ──
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge',  Ridge(alpha=1.0))
])
ridge_pipe.fit(X_train, y_train)
ridge_pred = ridge_pipe.predict(X_test)

# ── Lasso (alpha=1.0) ──
# WHY Pipeline for Lasso?
# Same reason as Ridge — StandardScaler must fit on training data only.
# Pipeline ensures scaler never sees test data during fitting.
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso',  Lasso(alpha=1.0, max_iter=10000))
    # WHY max_iter=10000?
    # Lasso uses coordinate descent (iterative) internally — not the Normal Equation.
    # With 30 features, it may need more iterations to converge.
    # Default is 1000 which sometimes throws a convergence warning.
])
lasso_pipe.fit(X_train, y_train)
lasso_pred = lasso_pipe.predict(X_test)

print("All three models trained.")

In [ ]:
# ── Step 4: The Key Question — How Many Features Did Each Model Keep? ──────────

lr_coefs    = lr.coef_
ridge_coefs = ridge_pipe.named_steps['ridge'].coef_
lasso_coefs = lasso_pipe.named_steps['lasso'].coef_

# Count non-zero coefficients
# WHY np.sum(coef != 0)?
# A coefficient is "kept" if it's non-zero.
# np.sum counts how many are non-zero (True=1, False=0)
lr_kept    = np.sum(lr_coefs    != 0)
ridge_kept = np.sum(np.abs(ridge_coefs) > 1e-10)  # tiny threshold for floating point
lasso_kept = np.sum(lasso_coefs != 0)

print("Features kept by each model (out of 30):")
print(f"  Linear Regression : {lr_kept} features")
print(f"  Ridge             : {ridge_kept} features")
print(f"  Lasso             : {lasso_kept} features  ← should be close to 5")

# Which features did Lasso keep?
lasso_kept_features = X.columns[lasso_coefs != 0].tolist()
print(f"\nFeatures Lasso kept: {lasso_kept_features}")
print(f"\nDid Lasso correctly identify the 5 real features?")
real_features = ['area_sqft', 'distance_from_station', 'floor_number', 'age_years', 'locality_score']
for feat in real_features:
    status = '✅ kept' if feat in lasso_kept_features else '❌ missed'
    print(f"  {feat:<25}: {status}")

In [ ]:
# ── Step 5: Performance Comparison ───────────────────────────────────────────
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return name, rmse, r2

results = [
    evaluate('Linear Regression', y_test, lr_pred),
    evaluate('Ridge (alpha=1.0)', y_test, ridge_pred),
    evaluate('Lasso (alpha=1.0)', y_test, lasso_pred),
]

print("=" * 52)
print("MODEL COMPARISON — MUMBAI FLAT PRICE")
print("=" * 52)
print(f"{'Model':<25} {'RMSE':>10} {'R²':>10}")
print("-" * 52)
for name, rmse, r2 in results:
    print(f"{name:<25} ₹{rmse:>7.2f}L  {r2:>8.4f}")
print("=" * 52)

In [ ]:
# ── Step 6: Visualise Coefficients — All Three Models ─────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

models_data = [
    ('Linear Regression (no penalty)', lr_coefs,    'steelblue'),
    ('Ridge — alpha=1.0 (L2)',         ridge_coefs, 'darkorange'),
    ('Lasso — alpha=1.0 (L1)',         lasso_coefs, 'green'),
]

for ax, (title, coefs, color) in zip(axes, models_data):
    colors = [color if c != 0 else 'lightgrey' for c in coefs]
    # WHY lightgrey for zero coefficients?
    # Makes it visually obvious which features Lasso eliminated
    ax.bar(X.columns, coefs, color=colors)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel('Coefficient')
    ax.tick_params(axis='x', rotation=45)
    non_zero = np.sum(coefs != 0)
    ax.set_xlabel(f'Features kept: {non_zero}/30')

plt.tight_layout()
plt.show()

# WHY this plot?
# Linear Regression: all 30 bars visible, noise features have small but non-zero values
# Ridge: all 30 bars visible but smaller, noise features very small but still there
# Lasso: only 5 real feature bars visible, 25 noise features are exactly zero (grey)

In [ ]:
# ── Step 7: Find Best Alpha with LassoCV ──────────────────────────────────────

# WHY not use test set to pick alpha?
# Using test set to make decisions = data leakage.
# Test set must be touched only ONCE — final evaluation.
# LassoCV uses 5-fold CV on training data only:
#   For each alpha candidate:
#     Split training data into 5 folds
#     Train on 4 folds, validate on 1 (rotate 5 times)
#     Average the 5 RMSE scores
#   Pick alpha with lowest average RMSE
#   Retrain on full training data with that best alpha

# Scale first — LassoCV needs scaled data
scaler   = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# WHY fit_transform only on X_train?
# Scaler learns mean and std from training data only.
# Applying it to test data later uses training statistics — no leakage.
X_test_scaled  = scaler.transform(X_test)
# WHY just transform (not fit_transform) on test?
# We don't refit — we use the same mean/std learned from training.

lasso_cv = LassoCV(
    alphas=np.logspace(-3, 2, 50),  # 50 alpha candidates from 0.001 to 100
    cv=5,                            # 5-fold cross-validation
    max_iter=10000,
    random_state=42
)
# WHY np.logspace(-3, 2, 50)?
# Alpha works on a logarithmic scale — the difference between 0.001 and 0.01
# matters as much as the difference between 1 and 10.
# logspace gives evenly spaced values on log scale — better coverage.

lasso_cv.fit(X_train_scaled, y_train)

best_alpha = lasso_cv.alpha_
# .alpha_ (with underscore) = best alpha found after CV

cv_pred = lasso_cv.predict(X_test_scaled)
cv_rmse = np.sqrt(mean_squared_error(y_test, cv_pred))
cv_r2   = r2_score(y_test, cv_pred)
cv_kept = np.sum(lasso_cv.coef_ != 0)

print(f"Best alpha found by LassoCV: {best_alpha:.4f}")
print(f"Features kept: {cv_kept}/30")
print(f"RMSE : ₹{cv_rmse:.2f} lakhs")
print(f"R²   : {cv_r2:.4f}")

cv_kept_features = X.columns[lasso_cv.coef_ != 0].tolist()
print(f"\nFeatures kept: {cv_kept_features}")

In [ ]:
# ── Step 8: Lasso Coefficient Path ────────────────────────────────────────────
# Watch features drop off one by one as alpha increases

alphas_path = np.logspace(-2, 2, 100)  # 100 alpha values from 0.01 to 100
coef_paths  = []

for alpha in alphas_path:
    lasso_temp = Lasso(alpha=alpha, max_iter=10000)
    lasso_temp.fit(X_train_scaled, y_train)
    coef_paths.append(lasso_temp.coef_.copy())

coef_paths = np.array(coef_paths)
# coef_paths shape: (100 alphas, 30 features)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: All features
for i, feat in enumerate(X.columns):
    is_real  = feat in real_features
    color    = 'steelblue' if is_real else 'lightgrey'
    lw       = 2 if is_real else 0.8
    axes[0].plot(np.log10(alphas_path), coef_paths[:, i],
                 color=color, linewidth=lw,
                 label=feat if is_real else None)

axes[0].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
axes[0].axvline(x=np.log10(best_alpha), color='red', linestyle='--',
                linewidth=1.5, label=f'Best alpha={best_alpha:.3f}')
axes[0].set_xlabel('log₁₀(alpha)  →  stronger penalty →')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Lasso Coefficient Path\nBlue = real features, Grey = noise features')
axes[0].legend(fontsize=8)

# Plot 2: Only real features (zoomed in)
for feat in real_features:
    i = list(X.columns).index(feat)
    axes[1].plot(np.log10(alphas_path), coef_paths[:, i],
                 linewidth=2, label=feat, marker='')

axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
axes[1].axvline(x=np.log10(best_alpha), color='red', linestyle='--',
                linewidth=1.5, label=f'Best alpha')
axes[1].set_xlabel('log₁₀(alpha)  →  stronger penalty →')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso Path — Real Features Only\n(noise features already at zero)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# WHY two plots?
# Left: full picture — notice noise features (grey) hit zero first and stay there
# Right: zoomed on real features — see how they survive longer before eventually shrinking
# Red line = best alpha chosen by CV — right in the zone where noise is gone but real features survive

In [ ]:
# ── Step 9: From Scratch — Lasso Coordinate Descent ──────────────────────────
# Lasso cannot use the Normal Equation (|w| has no smooth derivative at w=0)
# It uses COORDINATE DESCENT — update one weight at a time, holding others fixed

# WHY can't Lasso use Normal Equation or standard gradient descent?
# The derivative of |w| doesn't exist at w=0 (the V-shape has a sharp corner)
# Standard gradient descent needs a smooth derivative everywhere
# Coordinate descent gets around this using the "soft thresholding" trick

def soft_threshold(value, threshold):
    """
    The core of Lasso coordinate descent.
    If |value| <= threshold: return 0 (coefficient eliminated)
    If value > threshold:    return value - threshold (shrink from right)
    If value < -threshold:   return value + threshold (shrink from left)
    """
    # WHY this function?
    # This is what gives Lasso the ability to reach exactly zero.
    # If the "natural" coefficient is small enough, soft_threshold returns 0.
    # Ridge has no equivalent — it can only shrink, never zero.
    if abs(value) <= threshold:
        return 0.0
    elif value > threshold:
        return value - threshold
    else:
        return value + threshold

# Demo — show what soft_threshold does
print("Soft Thresholding with threshold=2:")
test_values = [-5, -2.5, -2, -1, 0, 1, 2, 2.5, 5]
for v in test_values:
    result = soft_threshold(v, 2)
    print(f"  soft_threshold({v:>5}, 2) = {result:>5}  {'← zeroed out!' if result == 0 else ''}")

print("\nNotice: values between -2 and +2 become exactly 0")
print("This is what Ridge CANNOT do — Ridge only shrinks, never zeros.")

In [ ]:
# ── Step 10: Final Comparison — All Models ────────────────────────────────────
print("=" * 60)
print("FINAL MODEL COMPARISON — MUMBAI FLAT PRICE")
print("=" * 60)
print(f"{'Model':<30} {'Features':>8} {'RMSE':>8} {'R²':>8}")
print("-" * 60)

lr_rmse  = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2    = r2_score(y_test, lr_pred)

ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2   = r2_score(y_test, ridge_pred)

lasso_rmse_10 = np.sqrt(mean_squared_error(y_test, lasso_pred))
lasso_r2_10   = r2_score(y_test, lasso_pred)

rows = [
    ('Linear Regression',       30, lr_rmse,      lr_r2),
    ('Ridge (alpha=1.0)',        30, ridge_rmse,   ridge_r2),
    ('Lasso (alpha=1.0)',        lasso_kept, lasso_rmse_10, lasso_r2_10),
    ('Lasso (best CV alpha)',    cv_kept,    cv_rmse,       cv_r2),
]

for name, kept, rmse, r2 in rows:
    print(f"{name:<30} {kept:>7}/30  ₹{rmse:>5.2f}L  {r2:>6.4f}")

print("=" * 60)
print(f"\nConclusion: Lasso (best alpha) kept only {cv_kept} features")
print("while achieving comparable or better performance than Linear Regression.")
print("Meena now knows exactly which features matter — no guesswork needed.")

In [ ]:
# ── Step 11: Visualise Final Predictions ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, cv_pred, alpha=0.4, color='green', s=20)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (₹ lakhs)')
axes[0].set_ylabel('Predicted Price (₹ lakhs)')
axes[0].set_title(f'Lasso (best alpha={best_alpha:.3f}) — Actual vs Predicted')
axes[0].legend()

residuals = y_test - cv_pred
axes[1].scatter(cv_pred, residuals, alpha=0.4, color='darkorange', s=20)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Price (₹ lakhs)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot — Random scatter = good model')

plt.tight_layout()
plt.show()

---

## Summary Table

| | Lasso Regression |
|---|---|
| **Full name** | Least Absolute Shrinkage and Selection Operator |
| **Task** | Regression + automatic feature selection |
| **Loss** | $MSE + \lambda \sum |w_i|$ |
| **Penalty type** | L1 — sum of absolute values of coefficients |
| **Slope of penalty** | Always 1 (constant) — never weakens |
| **Effect on coefficients** | Shrinks AND sets some to **exactly zero** |
| **Features eliminated?** | ✅ Yes — automatic feature selection |
| **Key hyperparameter** | `alpha` (= λ) — controls penalty strength |
| **How it learns** | Coordinate descent (not Normal Equation — |w| not smooth at 0) |
| **Must scale first?** | ✅ Yes — StandardScaler mandatory |
| **How to pick alpha** | LassoCV with cross-validation |
| **Strength** | Eliminates useless features — simpler, more interpretable model |
| **Weakness** | Correlated features — picks one arbitrarily, zeros the other |
| **When to use** | Many features, most are noise/irrelevant |
| **When NOT to use** | Many correlated features → Ridge is better |

---

## Ridge vs Lasso — Full Comparison

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| Penalty | $\lambda \sum w_i^2$ | $\lambda \sum |w_i|$ |
| Slope of penalty | 2w (shrinks as w→0) | 1 (always constant) |
| Reaches exactly zero? | ❌ No | ✅ Yes |
| Feature elimination | ❌ No | ✅ Yes |
| Correlated features | ✅ Handles well | ❌ Picks one arbitrarily |
| Interpretability | Medium (all features kept) | High (only important ones kept) |
| Best for | Multicollinearity | Many irrelevant features |

---

## What's Next?

Both Ridge and Lasso assume a **linear relationship** between features and output.

But what if the relationship is curved? What if price doesn't increase linearly with area — maybe it accelerates?

**5.4 Polynomial Regression** handles non-linear relationships by adding higher-degree terms (area², area³) while still using the linear regression framework.

---

## Practice Task

Ravi works at a hospital in Kochi predicting **patient recovery time in days** after surgery. His dataset has **25 features** — but he suspects only a few actually matter:

Real features (actually matter):
- `age` — patient age
- `bmi` — body mass index  
- `surgery_duration_hrs` — how long the surgery took
- `blood_loss_ml` — blood lost during surgery

Noise features: 21 random variables that have no relationship with recovery time

**Your tasks:**

1. Create synthetic dataset of 800 patients (4 real + 21 noise features)
2. Train Linear Regression, Ridge, and Lasso — compare features kept
3. Use LassoCV to find the best alpha
4. Verify Lasso correctly identified the 4 real features
5. Plot the Lasso coefficient path
6. Compare RMSE of all models

In [ ]:
# YOUR CODE HERE

# Step 1: Create dataset (4 real + 21 noise features)

# Step 2: Train Linear Regression, Ridge, Lasso

# Step 3: LassoCV — find best alpha

# Step 4: Verify real features identified

# Step 5: Coefficient path plot

# Step 6: Compare RMSE